# 基于新闻文本的A股股价波动预测与因果关系初探

> 以沪深300指数为研究对象，检验新闻情感是否具有市场预测能力。
>
> 理论框架：Tetlock (2007) "Giving Content to Investor Sentiment"

## 分析流程
1. 数据采集（Tushare 行情 + AkShare/GDELT 新闻）
2. 文本处理（清洗 + 分词）
3. 情感分析（词典法 + FinBERT + LLM）
4. 特征工程（日度情感指数 + 衰减加权）
5. VAR 模型 + Granger 因果检验
6. 回归分析 + 稳健性检验

## 0. 环境准备

In [ ]:
import sys
import os
import logging
import warnings
warnings.filterwarnings('ignore')

# 添加项目根目录到 path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print('环境准备完成')

## 1. 数据采集

In [ ]:
from src.data_collector import collect_all_data
from src.utils import load_config

config = load_config()
print(f"目标指数: {config['data']['index_name']} ({config['data']['index_code']})")
print(f"日期范围: {config['data']['start_date']} ~ {config['data']['end_date']}")

# 采集数据
data = collect_all_data(config)

stock_df = data['stock']
news_df = data['news']

print(f"\n行情数据: {len(stock_df)} 条")
print(stock_df.head())
print(f"\n新闻数据: {len(news_df)} 条")
if not news_df.empty:
    print(news_df.head())

## 2. 文本处理

In [ ]:
from src.news_processor import NewsProcessor

processor = NewsProcessor(config)

if not news_df.empty:
    news_df = processor.process_dataframe(news_df)
    print(f"文本处理完成: {len(news_df)} 条")
    print("\n示例:")
    for i in range(min(3, len(news_df))):
        print(f"\n--- 新闻 {i+1} ---")
        print(f"标题: {news_df.iloc[i].get('title', 'N/A')}")
        print(f"清洗后: {news_df.iloc[i]['cleaned_text'][:100]}...")
        print(f"分词(前10): {news_df.iloc[i]['tokens_filtered'][:10]}")
else:
    print('无新闻数据，跳过多本处理')

## 3. 情感分析

In [ ]:
from src.sentiment_analyzer import run_sentiment_analysis

if not news_df.empty:
    # 运行情感分析（默认: both = FinBERT + LLM）
    # 如果 LLM 未配置，会自动跳过
    news_df = run_sentiment_analysis(news_df, config, method='both')
    
    print('情感分析完成')
    print(f"\n可用情感列: {[c for c in news_df.columns if 'score' in c or 'label' in c]}")
    
    # 情感分布统计
    for col in ['dict_score', 'finbert_score', 'llm_score']:
        if col in news_df.columns:
            print(f"\n{col} 统计:")
            print(news_df[col].describe())
else:
    print('无新闻数据，跳过情感分析')

## 4. 特征工程

In [ ]:
from src.feature_engineer import FeatureEngineer

fe = FeatureEngineer(config)

if not news_df.empty:
    # 使用词典法情感作为主要指标（baseline）
    merged_df = fe.build_features(stock_df, news_df, score_col='dict_score')
    
    print(f"特征构建完成: {len(merged_df)} 行 x {len(merged_df.columns)} 列")
    print(f"\n列名: {list(merged_df.columns)}")
    print(f"\n数据预览:")
    display(merged_df[['trade_date', 'close', 'return', 'avg_sentiment', 'news_count', 'avg_sentiment_decay']].head(10))
    
    # 保存到 processed
    merged_df.to_csv('../data/processed/merged_dataset.csv', index=False, encoding='utf-8-sig')
    print('\n已保存到 data/processed/merged_dataset.csv')
else:
    print('无新闻数据，无法构建特征')

## 5. 可视化

In [ ]:
from src.visualization import Visualizer

viz = Visualizer(config)

if not news_df.empty:
    # 行情走势
    viz.plot_stock_price(merged_df)
    
    # 情感指数
    viz.plot_sentiment_timeline(merged_df, 'avg_sentiment')
    
    # 情感 vs 收益
    viz.plot_sentiment_vs_return(merged_df, 'avg_sentiment', lag=1)
    
    # 情感分布
    viz.plot_sentiment_distribution(merged_df, 'avg_sentiment')
    
    # 综合仪表板
    viz.plot_dashboard(merged_df, 'avg_sentiment')
    
    print('可视化完成，结果保存在 results/figures/')

## 6. VAR 模型与 Granger 因果检验

In [ ]:
from src.analysis import run_full_analysis

if not news_df.empty:
    analysis_results = run_full_analysis(merged_df, config)
    
    # 打印 VAR 摘要
    if 'var' in analysis_results and 'summary' in analysis_results['var']:
        print(analysis_results['var']['summary'])
    
    # 打印 Granger 因果检验
    if 'var' in analysis_results and 'granger' in analysis_results['var']:
        print('\n=== Granger 因果检验 ===')
        for pair, gc in analysis_results['var']['granger'].items():
            sig = '***' if gc['f_pvalue'] < 0.01 else ('*' if gc['f_pvalue'] < 0.05 else '')
            print(f"  {pair}: p={gc['f_pvalue']:.4f} {sig}")
    
    # 脉冲响应图
    if 'irf' in analysis_results:
        viz.plot_impulse_response(analysis_results['irf']['orthogonal_irf'])
        print('\n脉冲响应图已保存')
else:
    print('无新闻数据，跳过分析')

## 7. 回归分析结果汇总

In [ ]:
# 汇总关键回归结果
print('=== 情感→收益回归（核心结果）===\n')

for score_col in ['dict_score', 'finbert_score', 'llm_score']:
    key = f'regression_{score_col}'
    if key in analysis_results and 'summary' in analysis_results[key]:
        print(f'\n--- {score_col} ---')
        print(analysis_results[key]['summary'])

print('\n=== 收益→情感（反向因果检验）===\n')
for score_col in ['dict_score', 'finbert_score', 'llm_score']:
    key = f'reverse_regression_{score_col}'
    if key in analysis_results and 'summary' in analysis_results[key]:
        print(f'\n--- {score_col} ---')
        print(analysis_results[key]['summary'])

print('\n=== 极端情感→交易量 ===\n')
if 'regression_volume' in analysis_results and 'summary' in analysis_results['regression_volume']:
    print(analysis_results['regression_volume']['summary'])

## 8. 结论与讨论

### 核心发现
（根据实际运行结果填写）

### 与 Tetlock (2007) 的对比
- Tetlock 发现：悲观情绪预测次日下跌，一周内反转
- 本项目发现：...

### 因果性讨论
1. **伪因果风险**：宏观经济冲击可能同时影响新闻情绪和股价
2. **反向因果**：股价下跌 → 媒体更悲观 → 看似"悲观预测下跌"
3. **信息延迟**：新闻发布时间 vs 市场反应时间
4. **市场有效性**：如果情绪能预测收益，是否违背弱式有效市场假说？

### 局限性
- 新闻数据覆盖度有限
- 情感分析精度受模型限制
- 未控制所有混淆变量